# Phase AUG-VALIDATION: Augmented Top-Configs Retrain

**Defense prep.** The negative-transfer finding (full GLADA underperforms simpler ablations on ITW) was reported without MUSAN/RIR augmentation. IB-CAAN's 4.61% ITW EER used RawBoost + MUSAN + RIR; ours used RawBoost only. A reviewer or committee member will reasonably ask whether augmentation would rescue full GLADA.

**This notebook retrains three top configurations with full augmentation (RawBoost + MUSAN + RIR) and evaluates them on the same primary cross-domain target (ITW) plus the 2019 LA in-domain sanity check.** Three configs, default reduced epochs (3 Phase 1 + 17 Phase 2), single-shot runnable.

**Configs:**
1. `full_glada_aug` — wavlm-base-plus, full GLADA stack
2. `no_glada_aug` — wavlm-base-plus, single global DA (the baseline that beat full_glada at 14.89% vs 19.71% without aug)
3. `bb_wavlm-large_aug` — wavlm-large, full GLADA (best overall in original sweep at 12.64% without aug)

**Decision rule for the defense:**
- If `full_glada_aug` ITW EER < `no_glada_aug` ITW EER → negative-transfer claim is augmentation-confounded; flag in defense.
- If `full_glada_aug` ITW EER ≥ `no_glada_aug` ITW EER → negative-transfer claim is robust to augmentation; defense talking point.


## 1. Environment Setup


In [1]:
# Install dependencies and mount Drive
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
    print(f'BFloat16 support: {torch.cuda.is_bf16_supported()}')
else:
    raise RuntimeError('No GPU detected — cannot run augmented training without one')


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB (85.1 GB)
BFloat16 support: True


## 2. Copy GLADA Package to Local Runtime


In [2]:
import shutil, os, sys

# Drive package source — match the path used by the april13 phase notebooks
DRIVE_PKG = "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada"
LOCAL_PKG_ROOT = '/content/glada_code'
LOCAL_PKG = os.path.join(LOCAL_PKG_ROOT, 'glada')

# Allow override if the user has the package elsewhere
if not os.path.isdir(DRIVE_PKG):
    alt = '/content/drive/MyDrive/Jakob/glada_pkg'
    if os.path.isdir(alt):
        DRIVE_PKG = alt
        LOCAL_PKG = '/content/glada'
        LOCAL_PKG_ROOT = '/content'
    else:
        raise FileNotFoundError(
            f'Could not find glada package at either:\n'
            f'  {DRIVE_PKG}\n'
            f'  {alt}\n'
            f'Edit the DRIVE_PKG path above to point to your package.'
        )

# Fresh copy to avoid stale modules
if os.path.exists(LOCAL_PKG):
    shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)

if LOCAL_PKG_ROOT not in sys.path:
    sys.path.insert(0, LOCAL_PKG_ROOT)

print(f'Package copied: {DRIVE_PKG} -> {LOCAL_PKG}')
print(f'Contents: {sorted(os.listdir(LOCAL_PKG))}')


Package copied: /content/drive/MyDrive/Jakob/glada_pkg -> /content/glada
Contents: ['glada', 'pyproject.toml']


In [8]:
import shutil, os, sys

# Probe candidate paths for the package directory that contains __init__.py.
# The actual package dir must directly contain __init__.py; we then copy it
# to /content/glada and add /content to sys.path so `import glada` works.
CANDIDATES = [
    "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada",
    '/content/drive/MyDrive/Jakob/glada_pkg/glada',
    '/content/drive/MyDrive/Jakob/glada_pkg',
    '/content/drive/MyDrive/Jakob/glada',
]
DRIVE_PKG = next((c for c in CANDIDATES if os.path.exists(os.path.join(c, '__init__.py'))), None)
if DRIVE_PKG is None:
    raise FileNotFoundError(
        'No glada/__init__.py found in candidates. Run:\n'
        "  !find /content/drive -name '__init__.py' -path '*/glada/*' 2>/dev/null | head\n"
        'and add the parent directory to CANDIDATES above.'
    )

LOCAL_PKG = '/content/glada'
if os.path.exists(LOCAL_PKG): shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)

# Drop any stale glada module cache (in case a prior cell already imported)
for mod in list(sys.modules):
    if mod == 'glada' or mod.startswith('glada.'): del sys.modules[mod]
sys.path = [p for p in sys.path if 'glada' not in p]
if '/content' not in sys.path: sys.path.insert(0, '/content')
print(f'Package source: {DRIVE_PKG}\nCopied to:      {LOCAL_PKG}')


Package source: /content/drive/MyDrive/Jakob/glada_pkg/glada
Copied to:      /content/glada


## 3. Download Datasets and Augmentation Corpora


In [3]:
# This cell extracts ASVspoof 2019 LA and ITW from Drive, and downloads
# MUSAN + RIR from OpenSLR if not already present on Drive.
import tarfile, os

def extract_dataset(tar_path, extract_to, name):
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'{name}: already extracted to {extract_to}')
        return
    os.makedirs(extract_to, exist_ok=True)
    print(f'{name}: extracting {tar_path} -> {extract_to} ...')
    with tarfile.open(tar_path) as t:
        t.extractall(extract_to)
    print(f'{name}: done.')

DRIVE_ASVSPOOF_TAR = '/content/drive/MyDrive/Jakob/Datasets/LA.tar'
DRIVE_ITW_TAR      = '/content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar'

extract_dataset(DRIVE_ASVSPOOF_TAR, '/content/data/asvspoof', 'ASVspoof 2019 LA')
extract_dataset(DRIVE_ITW_TAR,      '/content/data/itw',      'In-the-Wild')

# --- MUSAN + RIR ---
MUSAN_PATH = '/content/data/musan'
RIR_PATH   = '/content/data/RIRS_NOISES/simulated_rirs'

MUSAN_DRIVE = '/content/drive/MyDrive/Jakob/Datasets/musan.tar.gz'
RIR_DRIVE   = '/content/drive/MyDrive/Jakob/Datasets/rirs_noises.zip'

os.makedirs('/content/data/', exist_ok=True)

if not os.path.isdir(MUSAN_PATH):
    if os.path.isfile(MUSAN_DRIVE):
        print(f'Copying MUSAN from Drive...')
        !cp {MUSAN_DRIVE} /content/data/
        !tar -xzf /content/data/musan.tar.gz -C /content/data/
        !rm /content/data/musan.tar.gz
    else:
        print('Downloading MUSAN from OpenSLR (1.5 GB)...')
        !wget -q --show-progress https://www.openslr.org/resources/17/musan.tar.gz -P /content/data/
        !tar -xzf /content/data/musan.tar.gz -C /content/data/
        !rm /content/data/musan.tar.gz
    print(f'MUSAN extracted to {MUSAN_PATH}')
else:
    print(f'MUSAN already present at {MUSAN_PATH}')

if not os.path.isdir(RIR_PATH):
    if os.path.isfile(RIR_DRIVE):
        print(f'Copying RIR from Drive...')
        !cp {RIR_DRIVE} /content/data/
        !unzip -q /content/data/rirs_noises.zip -d /content/data/
        !rm /content/data/rirs_noises.zip
    else:
        print('Downloading RIR from OpenSLR (~900 MB)...')
        !wget -q --show-progress https://www.openslr.org/resources/28/rirs_noises.zip -P /content/data/
        !unzip -q /content/data/rirs_noises.zip -d /content/data/
        !rm /content/data/rirs_noises.zip
    print(f'RIR extracted to {RIR_PATH}')
else:
    print(f'RIR already present at {RIR_PATH}')

# --- Verify all paths ---
ASV_PROTO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META       = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO      = '/content/data/itw/release_in_the_wild'

for p in [ASV_PROTO, ASV_AUDIO, ASV_EVAL_PROTO, ASV_EVAL_AUDIO, ITW_META, MUSAN_PATH, RIR_PATH]:
    assert os.path.exists(p), f'Missing: {p}'
print('\nAll paths verified — datasets and augmentation corpora ready.')


ASVspoof 2019 LA: extracting /content/drive/MyDrive/Jakob/Datasets/LA.tar -> /content/data/asvspoof ...


/tmp/ipykernel_537/4113359117.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(extract_to)


ASVspoof 2019 LA: done.
In-the-Wild: extracting /content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar -> /content/data/itw ...
In-the-Wild: done.
Copying MUSAN from Drive...
MUSAN extracted to /content/data/musan
Copying RIR from Drive...
RIR extracted to /content/data/RIRS_NOISES/simulated_rirs

All paths verified — datasets and augmentation corpora ready.


## 4. Imports


In [9]:
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,
    ASVspoofEvalDataset,
    InTheWildDataset,
)
from glada.augmentation import CombinedAugmenter
from glada.trainers import run_sweep

import copy, gc, json, time
import torch

print(f'GLADA package loaded.')
print(f'Available backbones: {list(BACKBONE_REGISTRY.keys())}')


GLADA package loaded.
Available backbones: ['wavlm-large', 'xlsr-300m', 'data2vec-large', 'wav2vec2-robust', 'wavlm-base-plus', 'xlsr-1b']


## 5. Configuration: 3 Top Configs With Augmentation


In [10]:
# ============================================================================
# Each entry maps a config name to a dict of GLADAConfig field overrides.
# An empty dict means default full GLADA on the chosen backbone.
#
# Augmentation (RawBoost + MUSAN + RIR) is applied uniformly to all three
# configs by setting musan_path / rir_path on the base config below.
# ============================================================================

# --- Epoch budget (matches original ablation sweep for fair comparison) ---
USE_FULL_EPOCHS = False  # Set True for publication-quality results (~5h -> ~12h per config)
PHASE1 = 5  if USE_FULL_EPOCHS else 3
PHASE2 = 45 if USE_FULL_EPOCHS else 17

AUG_CONFIGS = {
    # 1. Full GLADA + augmentation. The decisive comparison: did MUSAN+RIR rescue it?
    'full_glada_aug': {
        'backbone': 'wavlm-base-plus',
        'seed': 42,
    },
    # 2. The baseline that beat full_glada without augmentation (14.89% vs 19.71%).
    #    use_glada=False + use_global_domain_disc=True = single global DA, no per-group.
    'no_glada_aug': {
        'backbone': 'wavlm-base-plus',
        'seed': 42,
        'use_glada': False,
        'use_global_domain_disc': True,
    },
    # 3. Best-overall config (12.64% without aug). Stronger backbone, full stack.
    'bb_wavlm-large_aug': {
        'backbone': 'wavlm-large',
        'seed': 42,
    },
}

# --- Output directory on Drive ---
DRIVE_LOG_DIR = '/content/drive/MyDrive/Jakob/Experiments/aug_validation'
os.makedirs(DRIVE_LOG_DIR, exist_ok=True)

print(f'Configs to train: {len(AUG_CONFIGS)}')
for name, ovr in AUG_CONFIGS.items():
    print(f'  {name:<25s} {ovr}')
print(f'\nOutput dir: {DRIVE_LOG_DIR}')
print(f'Epoch budget: phase1={PHASE1}, phase2={PHASE2}')


Configs to train: 3
  full_glada_aug            {'backbone': 'wavlm-base-plus', 'seed': 42}
  no_glada_aug              {'backbone': 'wavlm-base-plus', 'seed': 42, 'use_glada': False, 'use_global_domain_disc': True}
  bb_wavlm-large_aug        {'backbone': 'wavlm-large', 'seed': 42}

Output dir: /content/drive/MyDrive/Jakob/Experiments/aug_validation
Epoch budget: phase1=3, phase2=17


## 6. Build Datasets (shared across all 3 configs)


In [11]:
# Build ONE base config that carries the augmentation paths, then construct
# CombinedAugmenter from it. The CombinedAugmenter:
#   - applies RawBoost with prob 0.5 (because use_rawboost=True is default)
#   - applies MUSAN noise with prob 0.5 (because musan_path is set)
#   - applies RIR convolution with prob 0.5 (because rir_path is set)
# The order mirrors the physical recording chain: codec -> noise -> reverb.

base_for_aug = GLADAConfig(
    batch_size=64,
    num_workers=0,           # Always 0 on Colab
    musan_path=MUSAN_PATH,
    rir_path=RIR_PATH,
    musan_prob=0.5,
    rir_prob=0.5,
    rawboost_prob=0.5,
    use_rawboost=True,
)

augmenter = CombinedAugmenter(base_for_aug)
print(f'Augmenter constructed:')
print(f'  RawBoost:  {augmenter.use_rawboost} (p={augmenter.rawboost_prob})')
print(f'  MUSAN:     {augmenter.musan_augmenter is not None} (p={augmenter.musan_prob})')
print(f'  RIR:       {augmenter.rir_augmenter is not None} (p={augmenter.rir_prob})')

# Source domain: ASVspoof 2019 LA training (labeled + speaker IDs).
# The augmenter is applied to the source-domain training audio only.
source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO,
    audio_dir=ASV_AUDIO,
    config=base_for_aug,
    augmenter=augmenter,
)

# Target domain: ITW (unlabeled for adversarial adaptation). No augmentation
# on target — we want the discriminator to see the real distribution.
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=base_for_aug,
    use_labels=False,
)

# Eval: ITW with labels, no augmentation.
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=base_for_aug,
    use_labels=True,
)

# In-domain sanity check: ASVspoof 2019 LA eval set, no augmentation.
extra_eval = {
    '2019_LA': ASVspoofEvalDataset(
        protocol_path=ASV_EVAL_PROTO,
        audio_dir=ASV_EVAL_AUDIO,
        config=base_for_aug,
    ),
}

num_speakers = len(source_ds.speaker_to_id) if hasattr(source_ds, 'speaker_to_id') else None
print(f'\nSource: {len(source_ds):,} samples, {num_speakers} speakers (augmented)')
print(f'Target: {len(target_ds):,} samples (unlabeled, not augmented)')
print(f'Eval (ITW):     {len(eval_ds):,} samples')
print(f'Eval (2019 LA): {len(extra_eval["2019_LA"]):,} samples')


Augmenter constructed:
  RawBoost:  True (p=0.5)
  MUSAN:     True (p=0.5)
  RIR:       True (p=0.5)
  ASVspoof: 25380 samples, 20 speakers, rawboost=0.5
  ITW: 31779 samples (unlabeled returned)
  ITW: 31779 samples (labeled returned)
  ASVspoof Eval: 71237 samples

Source: 25,380 samples, 20 speakers (augmented)
Target: 31,779 samples (unlabeled, not augmented)
Eval (ITW):     31,779 samples
Eval (2019 LA): 71,237 samples


## 7. Build Per-Config GLADAConfig Objects


In [12]:
# For each entry in AUG_CONFIGS, deep-copy base_for_aug and apply overrides.
# This produces a list of GLADAConfig objects ready for run_sweep.

configs = []
for name, ovr in AUG_CONFIGS.items():
    cfg = copy.deepcopy(base_for_aug)
    cfg.experiment_name = name
    cfg.phase1_epochs = PHASE1
    cfg.phase2_epochs = PHASE2
    cfg.drive_log_dir = DRIVE_LOG_DIR
    cfg.run_id = time.strftime('%Y%m%d_%H%M%S')
    for k, v in ovr.items():
        setattr(cfg, k, v)
    # Re-run __post_init__ logic that depends on backbone (groups, hidden_dim, ...)
    cfg.__post_init__()
    configs.append(cfg)
    print(f'{cfg.experiment_name:<25s} backbone={cfg.backbone:<18s} '
          f'glada={cfg.use_glada} global={cfg.use_global_domain_disc} '
          f'groups={cfg.num_groups}')


full_glada_aug            backbone=wavlm-base-plus    glada=True global=False groups=3
no_glada_aug              backbone=wavlm-base-plus    glada=False global=True groups=3
bb_wavlm-large_aug        backbone=wavlm-large        glada=True global=False groups=4


## 8. Run the Sweep

Three configs × ~5 hours each on A100 ≈ 15 hours total at the reduced epoch budget. Set `USE_FULL_EPOCHS = True` in cell 5 for publication-quality runs (will need a Pro+ session or split across days).


In [ ]:
data_paths = {
    'source_ds': source_ds,
    'target_ds': target_ds,
    'eval_ds': eval_ds,
    'extra_eval': extra_eval,
}

results = run_sweep(
    configs=configs,
    data_paths=data_paths,
    device='cuda',
)

print(f'\nSweep complete: {len(results)} configs trained.')
for name, res in results.items():
    if 'error' in res:
        print(f'  {name}: FAILED — {res["error"]}')
    else:
        print(f'  {name}: best ITW EER = {res.get("best_eer", "?"):.2f}%')



Config 1/3: wavlmbaseplus_s42_frz3_vb0.01_blr5e-06_llrd0.9
  backbone=wavlm-base-plus, seed=42, groups=3, phase1=3e, total=20e
  glada=True, vib=True, spk=True, cka=True, hier=True, rb=True
  Elapsed: 0.0 min | ETA: 0.0 min
  Updating num_speakers: 107 -> 20

Building GLADA model: wavlm-base-plus


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

  Trainable: 6,356,140 / 100,738,076
  Run dir: /content/drive/MyDrive/Jakob/Experiments/aug_validation/full_glada_aug/20260505_194531

GLADA Training | wavlm-base-plus | seed=42
  Phase 1: 3 ep | Phase 2: 17 ep
  Loss Weights: D=0.1, KL=0.01, SPK=0.1, CKA=0.01
  KL Annealing: 0 -> 0.01 over 5 epochs
  Phase 2 Adv Ramp: 0.0 -> 1.0 over 3 P2 epochs (+CKA)


... (179 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  [WARN] KL clamped 6000x consecutive (raw=2.061e+04, clamp=10.0)


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  Ep 16 P2: L=0.432 KL=10.00 | ITW=19.48% | 2019_LA=5.44%
  Early stopping at epoch 16

Done in 288.0 min | Best ITW EER: 12.03% (Ep 1)

Sweep complete: 3 configs in 752.9 min
Results saved to /content/drive/MyDrive/Jakob/Experiments/aug_validation/sweeps/sweep_results.json

Sweep complete: 3 configs trained.
  wavlmbaseplus_s42_frz3_vb0.01_blr5e-06_llrd0.9: best ITW EER = 14.18%
  wavlmbaseplus_s42_frz3_no-glada_globalDA_vb0.01_blr5e-06_llrd0.9: best ITW EER = 13.55%
  wavlmlarge_s42_frz3_vb0.01_blr5e-06_llrd0.9: best ITW EER = 12.03%


## 9. Decision Table

Compares the augmented runs against the original (no-aug) numbers from `Coding/results.json`. The decision rule from the title block:
- If `full_glada_aug` < `no_glada_aug` on ITW → augmentation rescued GLADA, flag in defense.
- Otherwise → negative-transfer claim is robust to augmentation.


In [ ]:
# Original numbers from Coding/results.json (no augmentation, same epoch budget)
ORIGINAL = {
    'full_glada':         {'ITW': 19.71, '2019_LA': 1.04},
    'no_glada':           {'ITW': 14.89, '2019_LA': 1.36},
    'bb_wavlm-large_s42': {'ITW': 12.64, '2019_LA': 4.08},
}
AUG_TO_ORIGINAL = {
    'full_glada_aug':     'full_glada',
    'no_glada_aug':       'no_glada',
    'bb_wavlm-large_aug': 'bb_wavlm-large_s42',
}

def fmt(v, width=10, prec=2):
    return f'{v:>{width}.{prec}f}' if isinstance(v, (int, float)) else f'{"---":>{width}s}'

header = f'{"config":<25s} {"orig ITW":>10s} {"aug ITW":>10s} {"Δ ITW":>10s}  {"orig 19LA":>10s} {"aug 19LA":>10s}'
print(header); print('-' * len(header))

table_rows = []
for aug_name, orig_name in AUG_TO_ORIGINAL.items():
    res = results.get(aug_name, {})
    if 'error' in res:
        print(f'{aug_name:<25s} FAILED: {res["error"]}')
        continue
    aug_itw  = res.get('eval_metrics', {}).get('ITW',     {}).get('eer', res.get('best_eer'))
    aug_la   = res.get('eval_metrics', {}).get('2019_LA', {}).get('eer')
    orig_itw = ORIGINAL[orig_name]['ITW']
    orig_la  = ORIGINAL[orig_name]['2019_LA']
    delta    = (aug_itw - orig_itw) if isinstance(aug_itw, (int, float)) else None
    print(f'{aug_name:<25s} {fmt(orig_itw)} {fmt(aug_itw)} {fmt(delta)}  {fmt(orig_la)} {fmt(aug_la)}')
    table_rows.append({
        'config': aug_name,
        'orig_itw_eer': orig_itw,
        'aug_itw_eer': aug_itw,
        'delta_itw': delta,
        'orig_2019LA_eer': orig_la,
        'aug_2019LA_eer': aug_la,
    })

# Save the comparison table to Drive
out_path = os.path.join(DRIVE_LOG_DIR, 'aug_validation_summary.json')
with open(out_path, 'w') as f:
    json.dump({'rows': table_rows, 'configs': list(AUG_CONFIGS.keys())}, f, indent=2, default=str)
print(f'\nSummary saved to: {out_path}')

# --- Verdict ---
if all(r['aug_itw_eer'] is not None for r in table_rows):
    by_name = {r['config']: r for r in table_rows}
    full_eer = by_name['full_glada_aug']['aug_itw_eer']
    no_eer   = by_name['no_glada_aug']['aug_itw_eer']
    print('\n' + '=' * 60)
    if full_eer < no_eer:
        print(f'VERDICT: full_glada_aug ({full_eer:.2f}%) BEAT no_glada_aug ({no_eer:.2f}%)')
        print('  -> Augmentation reverses the negative-transfer finding.')
        print('  -> Discuss in defense; original conclusion was confounded.')
    else:
        print(f'VERDICT: full_glada_aug ({full_eer:.2f}%) DID NOT beat no_glada_aug ({no_eer:.2f}%)')
        print('  -> Negative-transfer claim is robust to augmentation.')
        print('  -> Defense talking point: per-group adversarial DA hurts EVEN with strong augmentation.')
    print('=' * 60)


config                      orig ITW    aug ITW      Δ ITW   orig 19LA   aug 19LA
---------------------------------------------------------------------------------
full_glada_aug                 19.71        ---        ---        1.04        ---
no_glada_aug                   14.89        ---        ---        1.36        ---
bb_wavlm-large_aug             12.64        ---        ---        4.08        ---

Summary saved to: /content/drive/MyDrive/Jakob/Experiments/aug_validation/aug_validation_summary.json


## 10. (Optional) Full 6-Dataset Evaluation

The cells above evaluate on ITW + 2019 LA only (the primary cross-domain target plus an in-domain sanity check). For the full 6-dataset table matching the ICTAI paper, run `glada_comprehensive_eval.ipynb` against the checkpoints saved under `/content/drive/MyDrive/Jakob/Experiments/aug_validation/<run_id>/`.

The primary defense question is answered by the ITW comparison above; the 6-dataset eval is a nice-to-have if there's training time left.
